# GIZMO — Graph-Integrated Zone of Metabolite Operations
## Walkthrough Notebook

This notebook demonstrates the full GIZMO pipeline:

1. **Quick demo** — a few focused pathways
2. **Full human biology** — all 16,200 Reactome human reactions (~3 min first run, cached thereafter)
3. **Map Metabolon compounds** → ChEBI IDs via MetaNetX
4. **Flag currency metabolites** (ATP, NAD+, water, etc.)
5. **Run computational readiness QC**
6. **Load disease ontologies** (MONDO, Orphanet) + Open Targets
7. **Export** to GraphML / JSON
8. **Visualisation**

All data sources are CC BY 4.0. No HMDB or KEGG data is used.

In [1]:
import logging, sys
from pathlib import Path

repo_root = Path('.').resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import gizmo
print(f'GIZMO {gizmo.__version__}')

GIZMO 0.1.0


---
## Option A — Quick demo (a few pathways)

`ReactomeLoader.load_pathways()` handles:
- Mixed `int` / `dict` items in containedEvents (Reactome API quirk)
- 404 pathway IDs (silently skipped)
- Sub-pathway recursion
- Parallel fetching with local caching

In [2]:
from gizmo.sources.reactome import ReactomeLoader
from gizmo.graph.network import MetaboliteGraph
import pandas as pd

loader = ReactomeLoader(cache_dir=repo_root / 'data/raw/reactome', max_workers=5)

DEMO_PATHWAYS = [
    'R-HSA-70171',   # Glycolysis
    'R-HSA-70263',   # TCA / citric acid cycle
    'R-HSA-8964208', # Phenylalanine metabolism (PAH / PKU)
    'R-HSA-352230',  # Amino acid transport
    'R-HSA-71032',   # Amino acid synthesis
    'R-HSA-556833',  # Metabolism of lipids
]

mg_demo = loader.load_pathways(DEMO_PATHWAYS)
print(mg_demo)

INFO gizmo.sources.reactome: Found 1079 reactions across 6 pathways
INFO gizmo.sources.reactome: Fetching 1079 reactions (1079 cached / 0 to download, 5 workers) …


/home/jgardner/.cache/pypoetry/virtualenvs/gizmo--AQvKCjT-py3.13/lib/python3.13/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

MetaboliteGraph(912 metabolites [0 currency], 1079 reactions, 0 diseases, 0 genes, 3603 edges)


---
## Option B — Full human biology

`ReactomeLoader.load_species()` does a BFS from all 29 top-level human pathways.

| Run | Time | Notes |
|---|---|---|
| First | ~3–5 min | Downloads + caches ~16,000 reaction JSON files |
| Subsequent | ~30 s | Reads from `data/raw/reactome/` cache |


In [3]:
from gizmo.sources.reactome import ReactomeLoader
from gizmo.graph.network import MetaboliteGraph
import pandas as pd

loader = ReactomeLoader(
    cache_dir=repo_root / 'data/raw/reactome',
    max_workers=5,   # Reactome rate-limits; increase if cached
)

mg = loader.load_species('Homo sapiens')
print(mg)

INFO gizmo.sources.reactome: Collecting reaction stIDs for 'Homo sapiens' via BFS …
INFO gizmo.sources.reactome: Found 15408 unique reactions
INFO gizmo.sources.reactome: Fetching 15408 reactions (15408 cached / 0 to download, 5 workers) …


INFO gizmo.sources.reactome: Done. MetaboliteGraph(2839 metabolites [0 currency], 15408 reactions, 0 diseases, 0 genes, 17376 edges)


MetaboliteGraph(2839 metabolites [0 currency], 15408 reactions, 0 diseases, 0 genes, 17376 edges)


In [4]:
# Switch to demo graph if full biology not loaded yet
# mg = mg_demo

g = mg.graph
pd.Series(mg.summary(), name='count').to_frame()

,count
metabolites,2839
reactions,15408
diseases,0
genes,0
currency_flagged,0
edges,17376


In [5]:
# Pathway and annotation coverage
from collections import Counter

all_pathways = []
for nid in mg.reaction_nodes():
    all_pathways.extend(g.nodes[nid].get('pathways') or [])

n_rxn = len(mg.reaction_nodes())
with_ec   = sum(1 for n in mg.reaction_nodes() if g.nodes[n].get('ec_numbers'))
with_gene = sum(1 for n in mg.reaction_nodes() if g.nodes[n].get('gene_symbols'))

print(f'Unique pathways referenced:   {len(Counter(all_pathways))}')
print(f'Reactions with EC number:     {with_ec}/{n_rxn} ({with_ec/n_rxn*100:.1f}%)')
print(f'Reactions with gene symbol:   {with_gene}/{n_rxn} ({with_gene/n_rxn*100:.1f}%)')

Unique pathways referenced:   29
Reactions with EC number:     4663/15408 (30.3%)
Reactions with gene symbol:   14999/15408 (97.3%)


---
## Map Metabolon compounds → ChEBI

> One-time: download MetaNetX flat files (~500 MB).

In [6]:
from gizmo.sources.metanetx import MetaNetXClient

mnx = MetaNetXClient(cache_dir=repo_root / 'data/raw/metanetx')
mnx.download()

INFO gizmo.sources.metanetx: Cache hit: /home/jgardner/GIZMO/data/raw/metanetx/chem_xref.tsv
INFO gizmo.sources.metanetx: Cache hit: /home/jgardner/GIZMO/data/raw/metanetx/reac_xref.tsv
INFO gizmo.sources.metanetx: Cache hit: /home/jgardner/GIZMO/data/raw/metanetx/chem_prop.tsv
INFO gizmo.sources.metanetx: Cache hit: /home/jgardner/GIZMO/data/raw/metanetx/reac_prop.tsv


In [7]:
from gizmo.sources.metabolon import MetabolonLoader

met_loader = MetabolonLoader(
    repo_root / 'gizmo/sources/metabolon_data_dictionary_PMC_OA_subset_4.14.2024.csv'
)
n_indexed = met_loader.load_metanetx_index(
    repo_root / 'data/raw/metanetx/chem_prop.tsv',
    repo_root / 'data/raw/metanetx/chem_xref.tsv',
)
print(f'MetaNetX InChIKey index: {n_indexed:,} entries')

met_nodes, met_report = met_loader.to_metabolite_nodes(api_fallback=False)
print(met_report)

mg.add_metabolites(met_nodes)
print(mg)

INFO gizmo.sources.metabolon: chem_xref: 188748 MNX→ChEBI, 0 PubChem→ChEBI entries
INFO gizmo.sources.metabolon: MetaNetX index: 166362 exact InChIKey, 132734 connectivity prefix entries


MetaNetX InChIKey index: 166,362 entries


INFO gizmo.sources.metabolon: Metabolon mapping: 5395 compounds | InChIKey(exact): 1521 | InChIKey(connectivity): 884 | PubChem(local): 0 | InChIKey(API): 0 | PubChem(API): 0 | Unmatched: 2990 | ChEBI coverage: 44.6%


Metabolon mapping: 5395 compounds | InChIKey(exact): 1521 | InChIKey(connectivity): 884 | PubChem(local): 0 | InChIKey(API): 0 | PubChem(API): 0 | Unmatched: 2990 | ChEBI coverage: 44.6%
MetaboliteGraph(7690 metabolites [0 currency], 15408 reactions, 0 diseases, 0 genes, 17376 edges)


In [ ]:
# Metabolon compounds connected to at least one reaction
met_in_graph = [
    n for n in mg.metabolite_nodes()
    if g.nodes[n].get('metabolon_name') and mg.neighbors_of_metabolite(n)
]
total_met = sum(1 for n in mg.metabolite_nodes() if g.nodes[n].get('metabolon_name'))
print(f'Metabolon compounds in graph:   {total_met}')
print(f'  connected to >=1 reaction:    {len(met_in_graph)} ({len(met_in_graph)/max(total_met,1)*100:.1f}%)')

---
## Flag currency metabolites

In [ ]:
from gizmo.analysis.currency import flag_currency_metabolites, noncurrency_subgraph

result = flag_currency_metabolites(mg, degree_threshold_k=3.0)
print(f'Canonical:   {len(result["canonical"])}')
print(f'Statistical: {len(result["statistical"])}')
print(f'Total:       {len(result["total"])}')

pd.DataFrame([
    {'node_id': n, 'name': g.nodes[n].get('name',''), 'how': 'canonical' if n in result['canonical'] else 'statistical'}
    for n in result['total']
]).sort_values(['how','name'])

In [ ]:
import matplotlib.pyplot as plt

degrees = []
for nid in mg.metabolite_nodes():
    rxn_n = {n for n in list(g.predecessors(nid))+list(g.successors(nid)) if g.nodes[n].get('node_type')=='reaction'}
    degrees.append((nid, len(rxn_n), g.nodes[nid].get('is_currency', False)))

deg_df = pd.DataFrame(degrees, columns=['node_id','degree','currency']).sort_values('degree', ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(deg_df)), deg_df['degree'],
       color=['#E05252' if c else '#4C8BE0' for c in deg_df['currency']], width=1)
ax.set_yscale('log')
ax.set_xlabel('Metabolite rank'); ax.set_ylabel('Reaction degree')
ax.set_title('Metabolite reaction-degree distribution  (red = currency)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#E05252',label='Currency'), Patch(color='#4C8BE0',label='Non-currency')])
plt.tight_layout(); plt.show()

top15 = deg_df.head(15).copy()
top15['name'] = top15['node_id'].map(lambda n: g.nodes[n].get('name', n))
top15[['name','degree','currency']]

In [ ]:
mg_clean = noncurrency_subgraph(mg)
print('Full: ', mg)
print('Clean:', mg_clean)

---
## Computational readiness QC

In [ ]:
from gizmo.analysis.qc import assess_readiness

report = assess_readiness(mg)
report.print_summary()

In [ ]:
# Dead-end metabolites — structural gaps
if report.dead_end_ids:
    pd.DataFrame([
        {'node_id': n, 'name': g.nodes[n].get('name',''), 'chebi': g.nodes[n].get('chebi_id','')}
        for n in report.dead_end_ids[:30]
    ])

---
## Disease ontologies

> One-time: MONDO OBO (~200 MB), Orphanet XML (~50 MB).

In [ ]:
from gizmo.sources.mondo import MondoClient

mondo = MondoClient(cache_dir=repo_root / 'data/raw/mondo')
mondo.download()
iem = mondo.load_iem_subset()
print(f'IEM diseases (MONDO): {len(iem)}')
mg.add_diseases(iem)

pd.DataFrame([
    {'MONDO': d.node_id, 'Name': d.name,
     'OMIM': ', '.join(d.xref_omim[:2]), 'Orphanet': ', '.join(d.xref_orphanet[:2])}
    for d in iem[:15]
])

In [ ]:
from gizmo.sources.orphanet import OrphanetClient

orphanet = OrphanetClient(cache_dir=repo_root / 'data/raw/orphanet')
orphanet.download()

orpha_genes, orpha_edges = orphanet.load_gene_associations(
    association_types={'Disease-causing germline mutation(s) in'}
)
print(f'Genes: {len(orpha_genes)}, edges: {len(orpha_edges)}')
mg.add_genes(orpha_genes)
mg.add_disease_edges(orpha_edges)
print(mg)

---
## Open Targets

In [ ]:
from gizmo.sources.open_targets import OpenTargetsClient
from gizmo.schema import DiseaseNode

ot = OpenTargetsClient()

IEM_EXAMPLES = [
    ('MONDO:0009861', 'Phenylketonuria'),
    ('MONDO:0009563', 'Maple syrup urine disease'),
    ('MONDO:0005607', 'Homocystinuria'),
    ('MONDO:0009825', 'Propionic acidemia'),
    ('MONDO:0009624', 'Methylmalonic aciduria'),
]

rows = []
for mondo_id, name in IEM_EXAMPLES:
    if mondo_id not in mg.graph:
        mg.add_disease(DiseaseNode(node_id=mondo_id, mondo_id=mondo_id, name=name,
                                   is_inborn_error_of_metabolism=True, is_rare=True))
    genes, edges = ot.gene_associations_for_disease(mondo_id, min_score=0.05, max_results=100)
    mg.add_genes(genes); mg.add_disease_edges(edges)
    rows.append({'Disease': name, 'Genes': len(genes),
                 'Top gene': genes[0].symbol if genes else '-',
                 'Top score': round(edges[0].score, 3) if edges else 0})
    print(f'  {name}: {len(genes)} genes')

pd.DataFrame(rows)

---
## Export

In [ ]:
from gizmo.export.json_export import write_json, read_json
from gizmo.export.graphml import write_graphml

out = repo_root / 'data/processed'
out.mkdir(parents=True, exist_ok=True)

write_json(mg,       out / 'gizmo_full.json')
write_graphml(mg,    out / 'gizmo_full.graphml')
write_graphml(mg_clean, out / 'gizmo_noncurrency.graphml')

for f in sorted(out.iterdir()):
    print(f'  {f.name:<38} {f.stat().st_size/1024/1024:.1f} MB')

In [ ]:
mg2 = read_json(out / 'gizmo_full.json')
assert mg2.summary() == mg.summary()
print('Round-trip OK:', mg2.summary())

---
## Visualisation

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, networkx as nx

g_c = mg_clean.graph
comps = sorted(nx.connected_components(g_c.to_undirected()), key=len, reverse=True)
largest = g_c.subgraph(comps[0]).copy() if comps else g_c

MAX_NODES = 400
if len(largest) > MAX_NODES:
    import random; random.seed(42)
    largest = largest.subgraph(random.sample(list(largest.nodes()), MAX_NODES)).copy()
    print(f'Subsampled to {MAX_NODES} nodes')

COLORS = {'metabolite':'#4C8BE0','reaction':'#50C878','disease':'#E05252','gene':'#F5A623'}
SIZES  = {'metabolite':80,'reaction':60,'disease':200,'gene':150}

nc = [COLORS.get(largest.nodes[n].get('node_type',''),'#aaa') for n in largest]
ns = [SIZES.get(largest.nodes[n].get('node_type',''),50) for n in largest]

fig, ax = plt.subplots(figsize=(14,10))
pos = nx.spring_layout(largest, seed=42, k=2.0)
nx.draw_networkx(largest, pos=pos, ax=ax, node_color=nc, node_size=ns,
                 edge_color='#ccc', arrows=True, arrowsize=6,
                 with_labels=False, alpha=0.85)

met_labels = {n: largest.nodes[n].get('name',n)[:18]
              for n in largest if largest.nodes[n].get('node_type')=='metabolite'}
nx.draw_networkx_labels(largest, pos, labels=met_labels, ax=ax, font_size=5)

ax.legend(handles=[mpatches.Patch(color=c,label=t.capitalize()) for t,c in COLORS.items()],
          loc='upper left', fontsize=9)
ax.set_title(f'GIZMO largest component — currency-filtered ({len(largest)} nodes)', fontsize=12)
ax.axis('off'); plt.tight_layout()
plt.savefig(out/'gizmo_graph.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Connectivity stats on the full clean graph
comps_all = sorted(nx.connected_components(mg_clean.graph.to_undirected()), key=len, reverse=True)
print(f'Components:   {len(comps_all)}')
print(f'Largest:      {len(comps_all[0])} nodes ({len(comps_all[0])/len(mg_clean.graph)*100:.1f}%)')
print(f'Singletons:   {sum(1 for c in comps_all if len(c)==1)}')

fig, ax = plt.subplots(figsize=(10,3))
ax.bar(range(min(30,len(comps_all))), [len(c) for c in comps_all[:30]], color='#4C8BE0')
ax.set_xlabel('Component rank'); ax.set_ylabel('Size (nodes)')
ax.set_title('Component size distribution (top 30)')
plt.tight_layout(); plt.show()

---
## Downstream interop

### R / igraph / GATOM
```r
library(igraph)
g <- read_graph('data/processed/gizmo_noncurrency.graphml', format='graphml')
met <- induced_subgraph(g, V(g)[V(g)$node_type == 'metabolite'])
# V(met)$log2FC <- your_omics_results[V(met)$metabolon_name]
```

### PyTorch Geometric
```python
from torch_geometric.utils import from_networkx
for n, d in mg_clean.graph.nodes(data=True):
    d['x'] = [float(d.get('node_type')=='metabolite'),
               float(d.get('is_currency', False)),
               float(d.get('mass') or 0)]
data = from_networkx(mg_clean.graph, group_node_attrs=['x'])
```

### CLI
```bash
gizmo qc --graph data/processed/gizmo_full.json
gizmo metabolon --csv gizmo/sources/metabolon_data_dictionary_PMC_OA_subset_4.14.2024.csv \
  --metanetx-prop data/raw/metanetx/chem_prop.tsv \
  --metanetx-xref data/raw/metanetx/chem_xref.tsv
```